# 01. Time Series Fundamentals

## Background

A time series is any sequence of measurements taken at successive, regularly spaced points in time. The distinguishing feature of time series data — compared to cross-sectional data — is temporal dependence: observations close in time tend to be similar. This violates the i.i.d. assumption underlying most classical machine learning theory.

Classical time series analysis, developed primarily in the 1970s through the Box-Jenkins ARIMA methodology, decomposed series into trend, seasonality, and residual components. The foundational insight was stationarity: a series whose statistical properties (mean, variance, autocorrelation) do not change over time. Non-stationary series must be differenced or detrended before ARIMA modeling.

STL (Seasonal and Trend decomposition using Loess) is the workhorse for decomposition: it iteratively fits robust local regressions to extract seasonal and trend components. Understanding what each component represents — and how much variance it explains — is the first diagnostic step in any forecasting project.

## What You'll Learn

- Autocorrelation and partial autocorrelation functions (ACF/PACF)
- Stationarity tests: ADF (Augmented Dickey-Fuller), KPSS
- STL decomposition: trend, seasonality, residual
- Time series cross-validation: why expanding window beats random split

## Why This Matters

Every modern forecasting architecture — from transformers to neural ODEs — builds on this foundation. A PatchTST or TFT that processes a non-stationary series without appropriate normalization will fail. Understanding decomposition tells you which patterns the model must learn and which you can handle analytically.


## Generating Synthetic Time Series

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, List

np.random.seed(42)

def generate_ts(n: int = 500, trend: float = 0.05,
                seasonal_period: int = 52, seasonal_amp: float = 10.0,
                noise_std: float = 3.0) -> pd.Series:
    '''Generate a realistic time series with trend + seasonality + noise.'''
    t = np.arange(n)
    trend_component = trend * t
    seasonal_component = seasonal_amp * np.sin(2 * np.pi * t / seasonal_period)
    noise = np.random.normal(0, noise_std, n)

    values = 100 + trend_component + seasonal_component + noise
    dates = pd.date_range('2018-01-01', periods=n, freq='W')
    return pd.Series(values, index=dates, name='value')

ts = generate_ts()
print(f'Time series: {len(ts)} weekly observations')
print(f'Date range: {ts.index[0].date()} to {ts.index[-1].date()}')
print(f'Mean: {ts.mean():.2f}')
print(f'Std: {ts.std():.2f}')
print(f'Min: {ts.min():.2f}, Max: {ts.max():.2f}')
print(f'\nFirst 5 observations:')
print(ts.head().to_string())


## Autocorrelation & Stationarity

In [ ]:
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

def test_stationarity(series: pd.Series, name: str = 'Series') -> dict:
    '''ADF + KPSS stationarity tests.'''
    # ADF: H0 = unit root (non-stationary). Reject H0 => stationary
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag='AIC')

    # KPSS: H0 = stationary. Reject H0 => non-stationary
    kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='ct')

    adf_stationary = adf_p < 0.05
    kpss_stationary = kpss_p > 0.05

    print(f'Stationarity Tests for {name}:')
    print(f'  ADF test:  stat={adf_stat:.4f}, p={adf_p:.4f} => {"Stationary" if adf_stationary else "NON-STATIONARY"}')
    print(f'  KPSS test: stat={kpss_stat:.4f}, p={kpss_p:.4f} => {"Stationary" if kpss_stationary else "NON-STATIONARY"}')

    conclusion = 'Stationary' if (adf_stationary and kpss_stationary) else 'Non-stationary'
    print(f'  Conclusion: {conclusion} (both tests agree: {adf_stationary == kpss_stationary})')
    return {'adf_p': adf_p, 'kpss_p': kpss_p, 'stationary': conclusion == 'Stationary'}

# Test the raw series
result = test_stationarity(ts, 'Raw Series')

# Difference to make stationary
ts_diff = ts.diff().dropna()
print()
result_diff = test_stationarity(ts_diff, 'First Difference')

# ACF values
acf_values = acf(ts, nlags=20, fft=True)
pacf_values = pacf(ts, nlags=20)

print(f'\nACF at lags 0-5: {[f"{v:.3f}" for v in acf_values[:6]]}')
print(f'ACF at lag 52 (seasonal): {acf_values[min(52, len(acf_values)-1)]:.3f}')
print(f'\nSignificant autocorrelation at early lags indicates trend/seasonality')


## STL Decomposition

In [ ]:
# STL decomposition
stl = STL(ts, period=52, robust=True)
stl_result = stl.fit()

trend = stl_result.trend
seasonal = stl_result.seasonal
residual = stl_result.resid

# Variance explained by each component
total_var = ts.var()
trend_var = trend.var()
seasonal_var = seasonal.var()
residual_var = residual.var()

print('STL Decomposition Results:')
print(f'  Total variance:    {total_var:.2f}')
print(f'  Trend variance:    {trend_var:.2f} ({trend_var/total_var*100:.1f}%)')
print(f'  Seasonal variance: {seasonal_var:.2f} ({seasonal_var/total_var*100:.1f}%)')
print(f'  Residual variance: {residual_var:.2f} ({residual_var/total_var*100:.1f}%)')

# Strength of trend and seasonality (Wang 2006)
Ft = max(0, 1 - residual_var / (trend + residual).var())
Fs = max(0, 1 - residual_var / (seasonal + residual).var())
print(f'\nStrength metrics (0=absent, 1=dominant):')
print(f'  Trend strength:    {Ft:.3f}')
print(f'  Seasonal strength: {Fs:.3f}')

print(f'\nInterpretation:')
print(f'  Ft={Ft:.2f}: {"Strong trend" if Ft > 0.6 else "Weak trend"}')
print(f'  Fs={Fs:.2f}: {"Strong seasonality" if Fs > 0.6 else "Weak seasonality"}')
print(f'  These inform model selection: high Ft -> needs trend handling')
print(f'  High Fs -> needs seasonal modeling (SARIMA, seasonal embedding)')


## Time Series Cross-Validation

In [ ]:
from sklearn.base import BaseEstimator
from typing import Iterator

def expanding_window_cv(series: pd.Series,
                         n_splits: int = 5,
                         horizon: int = 12,
                         min_train: int = 52) -> Iterator[Tuple]:
    '''Time series expanding-window cross-validation.

    Unlike random k-fold (which leaks future data), expanding window
    trains on all data up to split point and tests on next H steps.
    '''
    n = len(series)
    step = (n - min_train - horizon) // n_splits

    for i in range(n_splits):
        train_end = min_train + i * step
        test_start = train_end
        test_end = test_start + horizon

        if test_end > n: break

        train = series.iloc[:train_end]
        test = series.iloc[test_start:test_end]
        yield train, test

def naive_forecast(train: pd.Series, horizon: int) -> np.ndarray:
    '''Seasonal naive: repeat last year as forecast.'''
    period = 52
    forecast = train.iloc[-period:].values
    if len(forecast) < horizon:
        forecast = np.tile(forecast, (horizon // len(forecast) + 1))[:horizon]
    return forecast[:horizon]

def mean_absolute_percentage_error(actual: np.ndarray, predicted: np.ndarray) -> float:
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# Evaluate naive seasonal baseline via expanding window CV
mapes = []
for i, (train, test) in enumerate(expanding_window_cv(ts, n_splits=5, horizon=12, min_train=52)):
    forecast = naive_forecast(train, horizon=len(test))
    mape = mean_absolute_percentage_error(test.values, forecast)
    mapes.append(mape)
    print(f'  Fold {i+1}: train={len(train)} obs, test={len(test)} obs, MAPE={mape:.2f}%')

print(f'\nSeasonal Naive Baseline — Mean MAPE: {np.mean(mapes):.2f}%')
print('Any model must beat this to justify its complexity.')
